# 第3讲：数据质量、时空特征与探索性分析（学生操练）

        > 课程：《交通大数据分析与应用》  
        > 数据：课程模拟数据，不是实际监测数据  
        > 建议用时：20分钟

        ## 目标

        1. 完成四类质量问题计数并决定处理规则；
2. 修改SPEED_MAX，观察有效样本量和均值如何变化；
3. 输出质量报告和路段—小时热力图；

        代码可以直接运行；请按`TODO`修改参数、核对输出并完成解释。

## 1. Setup｜环境、路径与参数

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})
RANDOM_STATE = 42
print("环境已就绪；数据目录：", DATA_DIR.resolve())

## 2. 读取原始数据并建立质量标记

In [ ]:
raw = pd.read_csv(DATA_DIR / "traffic_15min_dirty.csv")
parsed_time = pd.to_datetime(raw["timestamp"], errors="coerce")
SPEED_MAX = 130  # TODO：改为120，观察有效样本量与均值
flags = pd.DataFrame(index=raw.index)
flags["invalid_time"] = parsed_time.isna()
flags["missing_key_measure"] = raw[["flow_15min", "speed_kmh", "occupancy_pct"]].isna().any(axis=1)
flags["invalid_flow"] = ~raw["flow_15min"].between(0, 300, inclusive="both")
flags["invalid_speed"] = ~raw["speed_kmh"].between(0, SPEED_MAX, inclusive="both")
flags["invalid_occupancy"] = ~raw["occupancy_pct"].between(0, 100, inclusive="both")
flags["duplicate_row"] = raw.duplicated()
display(flags.sum().rename("count").to_frame())

## 3. 形成可追溯的清洗结果

In [ ]:
valid_mask = ~flags.any(axis=1)
clean = raw.loc[valid_mask].copy()
clean["timestamp"] = pd.to_datetime(clean["timestamp"])
clean["quality_status"] = "valid"
print("原始行数", len(raw), "有效行数", len(clean), "剔除行数", (~valid_mask).sum())
quality_report = flags.sum().rename("count").reset_index(name="count").rename(columns={"index":"issue"})
quality_report.to_csv(OUTPUT_DIR / "lesson03_quality_report.csv", index=False)
clean.to_csv(OUTPUT_DIR / "lesson03_clean_data.csv", index=False)

## 4. 比较时空聚合结果

In [ ]:
clean["hour"] = clean["timestamp"].dt.hour
heat = clean.pivot_table(index="detector_id", columns="hour", values="speed_kmh", aggfunc="mean")
display(heat.round(1))
fig, ax = plt.subplots(figsize=(9, 3.5))
im = ax.imshow(heat, aspect="auto", cmap="RdYlGn", vmin=25, vmax=80)
ax.set_yticks(range(len(heat.index)), heat.index); ax.set_xlabel("Hour"); ax.set_title("Mean speed by detector and hour")
fig.colorbar(im, ax=ax, label="km/h"); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lesson03_space_time_heatmap.png"); plt.show()

## 5. 完成自检

In [ ]:
checks = {"保留原始行数记录": len(raw) >= len(clean), "清洗后无越界速度": clean["speed_kmh"].between(0, SPEED_MAX).all(), "质量报告已生成": (OUTPUT_DIR / "lesson03_quality_report.csv").exists()}
status = "PASS" if all(checks.values()) else "CHECK"
(OUTPUT_DIR / "自检结果.txt").write_text(status, encoding="utf-8")
print(status, checks)

## Checks｜当堂记录

        - 完成四类质量问题计数并决定处理规则
- 修改SPEED_MAX，观察有效样本量和均值如何变化
- 输出质量报告和路段—小时热力图

        **预期结果：** 质量问题计数表、清洗后数据、时空热力图。

        **完成标准：** 每一种删除或修正都有规则；原始数据未被覆盖；Notebook生成PASS。

        请在课堂记录中写下：改了什么参数、结果发生了什么变化、这个变化在交通问题中意味着什么。